# Tutorial: Stateful Video Tracking with SAM 3.1 Object Multiplex

**Audience:** Python and computer-vision developers who want to track every instance of one text-described concept through an MP4 video.

**Prerequisites:** Follow the native setup in [README.md](README.md): Python 3.12, compatible NVIDIA CUDA hardware and PyTorch, Meta SAM 3 pinned to `660a5e9e1b8b4c02c0ad97229b88a09a6e4ff5b7`, access to the gated `facebook/sam3.1` checkpoint, and `requirements.txt`. Audio preservation also needs `ffmpeg` and `ffprobe` on `PATH`.

This notebook runs the model on your own CUDA machine. For hosted inference from a CPU-only machine, use [the Meta Model API notebook](sam3_meta_api.ipynb), which has separate setup and account requirements.

**Learning goals:** Open a stateful SAM 3.1 session, add one text noun phrase, propagate persistent object IDs, and render the returned masks with OpenCV. Meta's PyTorch predictor runs inference; OpenCV reads, overlays, and writes the video.


## Outline

1. Verify the environment
2. Configure one video and one concept phrase
3. Build the Object Multiplex predictor
4. Run stateful propagation and render the result
5. Review pitfalls and try a prompt exercise


## 1. Verify the environment

The pinned Meta implementation uses CUDA for SAM 3.1 video inference. Run this notebook from the `SAM-3` folder after completing the README setup. The reusable module imports the SAM package lazily, so its OpenCV renderer and tests remain usable without downloading the checkpoint.


In [ ]:
from pathlib import Path

import cv2
import torch
from IPython.display import Video, display

from sam3_video_tracking import build_sam31_predictor, track_video

print(f"PyTorch: {torch.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## 2. Configure the input and prompt

Use one noun phrase, such as `person` or `person wearing a red shirt`. A comma-separated string is still one phrase; it is not a documented multi-label API. Use a separate session for an unrelated concept.

Set `PRESERVE_AUDIO=True` to retain source audio; this requires both `ffmpeg` and `ffprobe`, including when checking whether a source has an audio stream. Leave it `False` for a silent output without these tools.


In [ ]:
INPUT_VIDEO = Path("input.mp4")
OUTPUT_VIDEO = Path("output-sam31.mp4")
TEXT_PROMPT = "person"
OUTPUT_THRESHOLD = 0.5
PRESERVE_AUDIO = False  # True requires ffmpeg and ffprobe on PATH.

if not INPUT_VIDEO.is_file():
    raise FileNotFoundError(
        f"Place your source video at {INPUT_VIDEO.resolve()} or change INPUT_VIDEO."
    )


## 3. Build the SAM 3.1 predictor

`build_sam3_multiplex_video_predictor` is Meta's recommended SAM 3.1 entry point. The default capacity and bucket size are 16 objects. The builder downloads `facebook/sam3.1` automatically after Hugging Face access and authentication are configured. Set `use_fa3=False` if FlashAttention 3 is unavailable. The pinned Meta revision has a documented start-session mismatch ([issue #544](https://github.com/facebookresearch/sam3/issues/544)); the companion filters that one unsupported keyword and becomes a no-op once the upstream signature is fixed.


In [ ]:
predictor = build_sam31_predictor(
    max_num_objects=16,
    multiplex_count=16,
    compile_model=False,
    use_fa3=False,
)


## 4. Propagate IDs and render the video

The helper starts one session, adds the text prompt on frame zero, and streams the canonical `propagate_in_video` responses forward, including propagated frame zero. That matters because SAM 3.1 can filter unconfirmed hot-start objects using later evidence. The helper always closes the session. OpenCV renders each response immediately, so full-resolution masks do not accumulate in memory. It blends each returned mask only where that mask is active and draws a stable object ID.


In [ ]:
summary = track_video(
    predictor,
    INPUT_VIDEO,
    OUTPUT_VIDEO,
    TEXT_PROMPT,
    output_threshold=OUTPUT_THRESHOLD,
    alpha=0.45,
    blur_kernel=11,
    preserve_audio=PRESERVE_AUDIO,
)
summary


### Preview the rendered result

The output should preserve the input frame count and dimensions. Colors and labels identify SAM's object IDs; they are not ground-truth annotations.

The default OpenCV `mp4v` codec can produce a valid MP4 that a browser cannot play. If the preview is blank, set `MAKE_BROWSER_PREVIEW=True` below to create an H.264 copy with `ffmpeg`. This changes only the preview encoding.


In [ ]:
import shutil
import subprocess

MAKE_BROWSER_PREVIEW = False
preview_path = OUTPUT_VIDEO
if MAKE_BROWSER_PREVIEW:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("Install ffmpeg to create an H.264 browser preview.")
    preview_path = OUTPUT_VIDEO.with_name(f"{OUTPUT_VIDEO.stem}-preview.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(OUTPUT_VIDEO),
        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac",
        "-movflags", "+faststart", str(preview_path),
    ], check=True)

display(Video(str(preview_path), embed=False))


## Pitfalls and production guidance

- Do not call the image processor independently on every frame and label the result tracking; that path has no temporal memory or persistent identities.
- Do not treat commas as a list of labels. One text request represents one phrase.
- Stateful propagation mitigates drift and identity switches but does not guarantee perfect recovery through every occlusion. Validate thresholds on your own domain.
- Compilation adds startup cost. Warm up before timing, and report the GPU, object count, resolution, and software versions with performance results.
- The current predictor loads video frames into session state, so long or high-resolution videos require careful GPU and host-memory planning.


## Exercise

Choose a more specific phrase that still describes one concept, predict whether it will reduce or increase the number of tracked instances, and render it to a different output file. Do not append it to the first prompt with a comma.


In [ ]:
# Exercise answer scaffold
EXERCISE_PROMPT = "person wearing a red shirt"
EXERCISE_OUTPUT = Path("output-sam31-specific-prompt.mp4")

# Uncomment after recording your prediction. track_video opens and closes a new session.
# exercise_summary = track_video(
#     predictor, INPUT_VIDEO, EXERCISE_OUTPUT, EXERCISE_PROMPT,
#     preserve_audio=PRESERVE_AUDIO,
# )
# exercise_summary


## Primary references

- [Meta SAM 3.1 release notes](https://github.com/facebookresearch/sam3/blob/main/RELEASE_SAM3p1.md)
- [Meta SAM 3.1 Object Multiplex notebook](https://github.com/facebookresearch/sam3/blob/main/examples/sam3.1_video_predictor_example.ipynb)
- [SAM 3 paper](https://arxiv.org/abs/2511.16719)
